In [ ]:
# ============================================================
# Cell 1: Import required libraries and setup
# ============================================================
# This notebook estimates the unknown HBV model parameters using DINNs.
# This code uses 10 data points for parameter estimation
# The generated synthetic data are first saved and then loaded before training.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import odeint

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# For reproducibility
np.random.seed(1234)
torch.manual_seed(1234)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Working folder for saving and loading the generated synthetic data
WORK_DIR = "HBV_DINN_10_data_points"
os.makedirs(WORK_DIR, exist_ok=True)
print("Working directory:", WORK_DIR)

In [ ]:
# ============================================================
# Cell 2: Generate, save, and load simulated HBV data
# ============================================================
# The synthetic data are generated from the HBV ODE model.
# Then the data are saved as a CSV file and loaded again for DINN training.


# Initial conditions
X0 = 12.79e8
Y0 = 4.95e8
D0 = 8.04e10
V0 = 1.85e10

# Time grid
t = np.linspace(0, 500, 10)

# True parameter values
true_params = {
    "lamda": 2.6e7,
    "mu": 0.01,
    "k": 1.67e-12,
    "a": 150,
    "beta": 0.87,
    "delta": 0.053,
    "c": 3.8,
    "alpha": 0.8,
    "gamma": 0.6931
}

# HBV model
def hbv_model(y, t, lamda, mu, k, a, beta, delta, c, alpha, gamma):
    X, Y, D, V = y

    dXdt = lamda - mu * X - k * V * X
    dYdt = k * V * X - delta * Y
    dDdt = a * Y + gamma * (1 - alpha) * D - alpha * beta * D - delta * D
    dVdt = alpha * beta * D - c * V

    return [dXdt, dYdt, dDdt, dVdt]

# Initial condition vector
y0 = [X0, Y0, D0, V0]

# Solve ODE system
solution = odeint(
    hbv_model,
    y0,
    t,
    args=(
        true_params["lamda"],
        true_params["mu"],
        true_params["k"],
        true_params["a"],
        true_params["beta"],
        true_params["delta"],
        true_params["c"],
        true_params["alpha"],
        true_params["gamma"]
    )
)

# Separate simulated state variables
X, Y, D, V = solution.T

# Store data row-wise: [t, X, Y, D, V]
dataset_array = np.asarray([t, X, Y, D, V])

# Save generated synthetic data
DATA_FILE = os.path.join(WORK_DIR, "hbv_DINNs_10_data_points.csv")
np.savetxt(DATA_FILE, dataset_array, delimiter=",")

print("Synthetic data saved at:")
print(DATA_FILE)

# Load saved synthetic data
HBV_data = np.genfromtxt(DATA_FILE, delimiter=",")

# Extract data after loading
t_data = HBV_data[0]
X_data = HBV_data[1]
Y_data = HBV_data[2]
D_data = HBV_data[3]
V_data = HBV_data[4]

print("Synthetic HBV data loaded successfully.")
print("Shape of loaded data:", HBV_data.shape)
print("Number of time points:", len(t_data))

In [ ]:
# ============================================================
# Cell 3: Define the DINN model
# ============================================================
# alpha is fixed at 0.8 and is not estimated.


class DINN(nn.Module):
    def __init__(self, t, X_data, Y_data, D_data, V_data, alpha_fixed=0.8):
        super(DINN, self).__init__()

        # Fixed alpha
        self.alpha_fixed = torch.tensor(alpha_fixed, dtype=torch.float32, device=device)

        # Time tensor with gradient
        self.t = torch.tensor(t, dtype=torch.float32, device=device).view(-1, 1)
        self.t.requires_grad_(True)

        # Data tensors
        self.X = torch.tensor(X_data, dtype=torch.float32, device=device).view(-1, 1)
        self.Y = torch.tensor(Y_data, dtype=torch.float32, device=device).view(-1, 1)
        self.D = torch.tensor(D_data, dtype=torch.float32, device=device).view(-1, 1)
        self.V = torch.tensor(V_data, dtype=torch.float32, device=device).view(-1, 1)

        # Min-max values
        self.X_min, self.X_max = torch.min(self.X), torch.max(self.X)
        self.Y_min, self.Y_max = torch.min(self.Y), torch.max(self.Y)
        self.D_min, self.D_max = torch.min(self.D), torch.max(self.D)
        self.V_min, self.V_max = torch.min(self.V), torch.max(self.V)

        # Normalized data
        self.X_hat = (self.X - self.X_min) / (self.X_max - self.X_min)
        self.Y_hat = (self.Y - self.Y_min) / (self.Y_max - self.Y_min)
        self.D_hat = (self.D - self.D_min) / (self.D_max - self.D_min)
        self.V_hat = (self.V - self.V_min) / (self.V_max - self.V_min)

        # Neural network
        self.net_HBV = self.Net_HBV().to(device)

        self.lamda_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.mu_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.k_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.a_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.beta_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.delta_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.c_tilda = nn.Parameter(torch.zeros(1, device=device))
        self.gamma_tilda = nn.Parameter(torch.zeros(1, device=device))

        # History
        self.history = {
            "iteration": [],
            "total_loss": [],
            "data_loss": [],
            "residual_loss": [],
            "X_data_loss": [],
            "Y_data_loss": [],
            "D_data_loss": [],
            "V_data_loss": [],
            "f1_loss": [],
            "f2_loss": [],
            "f3_loss": [],
            "f4_loss": [],
            "lamda": [],
            "mu": [],
            "k": [],
            "a": [],
            "beta": [],
            "delta": [],
            "c": [],
            "alpha_fixed": [],
            "gamma": []
        }

        # Parameters to optimize
        self.params = list(self.net_HBV.parameters()) + [
            self.lamda_tilda,
            self.mu_tilda,
            self.k_tilda,
            self.a_tilda,
            self.beta_tilda,
            self.delta_tilda,
            self.c_tilda,
            self.gamma_tilda
        ]

    # --------------------------------------------------------
    # Parameter transformations: 10% range around references
    # p = p_ref + 0.1*p_ref*tanh(p_tilda)
    # --------------------------------------------------------

    @property
    def lamda(self):
        return torch.tanh(self.lamda_tilda) * 2.6e6 + 2.6e7

    @property
    def mu(self):
        return torch.tanh(self.mu_tilda) * 0.001 + 0.01

    @property
    def k(self):
        return torch.tanh(self.k_tilda) * 1.67e-13 + 1.67e-12

    @property
    def a(self):
        return torch.tanh(self.a_tilda) * 15 + 150

    @property
    def beta(self):
        return torch.tanh(self.beta_tilda) * 0.087 + 0.87

    @property
    def delta(self):
        return torch.tanh(self.delta_tilda) * 0.0053 + 0.053

    @property
    def c(self):
        return torch.tanh(self.c_tilda) * 0.38 + 3.8

    @property
    def alpha(self):
        return self.alpha_fixed

    @property
    def gamma(self):
        return torch.tanh(self.gamma_tilda) * 0.06931 + 0.6931

    class Net_HBV(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc1 = nn.Linear(1, 20)
            self.fc2 = nn.Linear(20, 20)
            self.fc3 = nn.Linear(20, 20)
            self.fc4 = nn.Linear(20, 20)
            self.out = nn.Linear(20, 4)

        def forward(self, t):
            z = F.relu(self.fc1(t))
            z = F.relu(self.fc2(z))
            z = F.relu(self.fc3(z))
            z = F.relu(self.fc4(z))
            z = self.out(z)
            return z

    def net_f(self):
        HBV_hat = self.net_HBV(self.t)

        X_hat = HBV_hat[:, 0:1]
        Y_hat = HBV_hat[:, 1:2]
        D_hat = HBV_hat[:, 2:3]
        V_hat = HBV_hat[:, 3:4]

        # Automatic differentiation
        X_hat_t = torch.autograd.grad(
            X_hat, self.t,
            grad_outputs=torch.ones_like(X_hat),
            retain_graph=True,
            create_graph=True
        )[0]

        Y_hat_t = torch.autograd.grad(
            Y_hat, self.t,
            grad_outputs=torch.ones_like(Y_hat),
            retain_graph=True,
            create_graph=True
        )[0]

        D_hat_t = torch.autograd.grad(
            D_hat, self.t,
            grad_outputs=torch.ones_like(D_hat),
            retain_graph=True,
            create_graph=True
        )[0]

        V_hat_t = torch.autograd.grad(
            V_hat, self.t,
            grad_outputs=torch.ones_like(V_hat),
            retain_graph=True,
            create_graph=True
        )[0]

        # Unnormalize predicted variables
        X = self.X_min + (self.X_max - self.X_min) * X_hat
        Y = self.Y_min + (self.Y_max - self.Y_min) * Y_hat
        D = self.D_min + (self.D_max - self.D_min) * D_hat
        V = self.V_min + (self.V_max - self.V_min) * V_hat

        # Normalized residuals
        f1_hat = X_hat_t - (
            self.lamda - self.mu * X - self.k * X * V
        ) / (self.X_max - self.X_min)

        f2_hat = Y_hat_t - (
            self.k * X * V - self.delta * Y
        ) / (self.Y_max - self.Y_min)

        f3_hat = D_hat_t - (
            self.a * Y + self.gamma * (1 - self.alpha) * D
            - self.alpha * self.beta * D - self.delta * D
        ) / (self.D_max - self.D_min)

        f4_hat = V_hat_t - (
            self.alpha * self.beta * D - self.c * V
        ) / (self.V_max - self.V_min)

        return f1_hat, f2_hat, f3_hat, f4_hat, X_hat, Y_hat, D_hat, V_hat

    def print_loss_values(self, epoch, total_loss, data_loss, residual_loss):
        print(
            f"Epoch {epoch:7d} | "
            f"Total loss = {total_loss.item():.6e} | "
            f"Data loss = {data_loss.item():.6e} | "
            f"Residual loss = {residual_loss.item():.6e}"
        )

    def train_model(self, n_epochs, optimizer, scheduler=None, print_every=10000):
        print("Starting DINN training with fixed alpha = 0.8 ...")

        for epoch in range(n_epochs):
            optimizer.zero_grad()

            f1, f2, f3, f4, X_pred_hat, Y_pred_hat, D_pred_hat, V_pred_hat = self.net_f()

            # Data loss
            X_data_loss = torch.mean((self.X_hat - X_pred_hat) ** 2)
            Y_data_loss = torch.mean((self.Y_hat - Y_pred_hat) ** 2)
            D_data_loss = torch.mean((self.D_hat - D_pred_hat) ** 2)
            V_data_loss = torch.mean((self.V_hat - V_pred_hat) ** 2)

            data_loss = X_data_loss + Y_data_loss + D_data_loss + V_data_loss

            # Residual loss
            f1_loss = torch.mean(f1 ** 2)
            f2_loss = torch.mean(f2 ** 2)
            f3_loss = torch.mean(f3 ** 2)
            f4_loss = torch.mean(f4 ** 2)

            residual_loss = f1_loss + f2_loss + f3_loss + f4_loss

            # Total loss
            total_loss = data_loss + residual_loss

            total_loss.backward()
            optimizer.step()

            if scheduler is not None:
                scheduler.step()

            # Store history
            self.history["iteration"].append(epoch)
            self.history["total_loss"].append(total_loss.item())
            self.history["data_loss"].append(data_loss.item())
            self.history["residual_loss"].append(residual_loss.item())

            self.history["X_data_loss"].append(X_data_loss.item())
            self.history["Y_data_loss"].append(Y_data_loss.item())
            self.history["D_data_loss"].append(D_data_loss.item())
            self.history["V_data_loss"].append(V_data_loss.item())

            self.history["f1_loss"].append(f1_loss.item())
            self.history["f2_loss"].append(f2_loss.item())
            self.history["f3_loss"].append(f3_loss.item())
            self.history["f4_loss"].append(f4_loss.item())

            self.history["lamda"].append(self.lamda.item())
            self.history["mu"].append(self.mu.item())
            self.history["k"].append(self.k.item())
            self.history["a"].append(self.a.item())
            self.history["beta"].append(self.beta.item())
            self.history["delta"].append(self.delta.item())
            self.history["c"].append(self.c.item())
            self.history["alpha_fixed"].append(self.alpha.item())
            self.history["gamma"].append(self.gamma.item())

            # Print only losses every 10000 iterations and also at final iteration
            if epoch % print_every == 0 or epoch == n_epochs - 1:
                self.print_loss_values(epoch, total_loss, data_loss, residual_loss)

        print("Training completed.")

    def predict(self):
        self.eval()

        with torch.no_grad():
            HBV_hat = self.net_HBV(self.t)

            X_hat = HBV_hat[:, 0:1]
            Y_hat = HBV_hat[:, 1:2]
            D_hat = HBV_hat[:, 2:3]
            V_hat = HBV_hat[:, 3:4]

            X_pred = self.X_min + (self.X_max - self.X_min) * X_hat
            Y_pred = self.Y_min + (self.Y_max - self.Y_min) * Y_hat
            D_pred = self.D_min + (self.D_max - self.D_min) * D_hat
            V_pred = self.V_min + (self.V_max - self.V_min) * V_hat

        return (
            X_pred.cpu().numpy().flatten(),
            Y_pred.cpu().numpy().flatten(),
            D_pred.cpu().numpy().flatten(),
            V_pred.cpu().numpy().flatten()
        )

In [ ]:
# ============================================================
# Cell 4: Initialize and train the DINN model
# ============================================================
# Initialize DINN
dinn = DINN(
    t_data,
    X_data,
    Y_data,
    D_data,
    V_data,
    alpha_fixed=0.8
).to(device)

# Optimizer
learning_rate = 1e-6
optimizer = optim.Adam(dinn.params, lr=learning_rate)

# Cyclic learning-rate scheduler
scheduler = torch.optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr=1e-6,
    max_lr=1e-3,
    step_size_up=1000,
    mode="exp_range",
    gamma=0.85,
    cycle_momentum=False
)

# Number of training epochs
N_EPOCHS = 1000000

# Train model
dinn.train_model(
    n_epochs=N_EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
    print_every=10000
)

In [ ]:
# ============================================================
# Cell 5: Print final estimated parameter values
# ============================================================
# alpha is fixed at 0.8.
# The remaining eight parameters are estimated.

final_params_df = pd.DataFrame({
    "parameter": [
        "lamda", "mu", "k", "a", "beta", "delta", "c", "alpha", "gamma"
    ],
    "true_value": [
        true_params["lamda"],
        true_params["mu"],
        true_params["k"],
        true_params["a"],
        true_params["beta"],
        true_params["delta"],
        true_params["c"],
        true_params["alpha"],
        true_params["gamma"]
    ],
    "estimated_or_fixed_value": [
        dinn.lamda.item(),
        dinn.mu.item(),
        dinn.k.item(),
        dinn.a.item(),
        dinn.beta.item(),
        dinn.delta.item(),
        dinn.c.item(),
        dinn.alpha.item(),
        dinn.gamma.item()
    ],
    "status": [
        "estimated",
        "estimated",
        "estimated",
        "estimated",
        "estimated",
        "estimated",
        "estimated",
        "fixed",
        "estimated"
    ]
})

print("Final estimated parameter values:")
print(final_params_df)

In [ ]:
# ============================================================
# Cell 6: Generate predictions


X_pred, Y_pred, D_pred, V_pred = dinn.predict()

print("Predictions generated successfully.")

In [ ]:
# ============================================================
# Cell 7: Plot true data and DINN predictions
# ============================================================

fig, axs = plt.subplots(1, 4, figsize=(28, 7))
fig.patch.set_facecolor("white")

ax1, ax2, ax3, ax4 = axs.flatten()

# X
ax1.scatter(t_data, X_data, color="red", alpha=0.9, label="Uninfected data", s=45)
ax1.plot(t_data, X_pred, color="blue", lw=4, label="Uninfected prediction")
ax1.set_title("Uninfected hepatocytes", fontsize=26)
ax1.set_xlabel("Time (days)", fontsize=28)
ax1.set_ylabel(r"Concentration of $X$", fontsize=28)
ax1.legend(fontsize=18)
ax1.tick_params(axis="both", labelsize=20)
ax1.yaxis.get_offset_text().set_fontsize(20)

# Y
ax2.scatter(t_data, Y_data, color="red", alpha=0.9, label="Infected data", s=45)
ax2.plot(t_data, Y_pred, color="blue", lw=4, label="Infected prediction")
ax2.set_title("Infected hepatocytes", fontsize=26)
ax2.set_xlabel("Time (days)", fontsize=28)
ax2.set_ylabel(r"Concentration of $Y$", fontsize=28)
ax2.legend(fontsize=18)
ax2.tick_params(axis="both", labelsize=20)
ax2.yaxis.get_offset_text().set_fontsize(20)

# D
ax3.scatter(t_data, D_data, color="red", alpha=0.9, label="Capsid data", s=45)
ax3.plot(t_data, D_pred, color="blue", lw=4, label="Capsid prediction")
ax3.set_title("Capsids", fontsize=26)
ax3.set_xlabel("Time (days)", fontsize=28)
ax3.set_ylabel(r"Concentration of $D$", fontsize=28)
ax3.legend(fontsize=18)
ax3.tick_params(axis="both", labelsize=20)
ax3.yaxis.get_offset_text().set_fontsize(20)

# V
ax4.scatter(t_data, V_data, color="red", alpha=0.9, label="Virion data", s=45)
ax4.plot(t_data, V_pred, color="blue", lw=4, label="Virion prediction")
ax4.set_title("Virions", fontsize=26)
ax4.set_xlabel("Time (days)", fontsize=28)
ax4.set_ylabel(r"Concentration of $V$", fontsize=28)
ax4.legend(fontsize=18)
ax4.tick_params(axis="both", labelsize=20)
ax4.yaxis.get_offset_text().set_fontsize(20)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 8: Calculate and print RRMSE errors
# ============================================================
# RRMSE = sqrt(sum((true - predicted)^2) / sum(true^2))


def rrmse_error(true, pred):
    return np.sqrt(np.sum((true - pred) ** 2) / np.sum(true ** 2))

X_RRMSE = rrmse_error(X_data, X_pred)
Y_RRMSE = rrmse_error(Y_data, Y_pred)
D_RRMSE = rrmse_error(D_data, D_pred)
V_RRMSE = rrmse_error(V_data, V_pred)

rrmse_df = pd.DataFrame({
    "variable": ["X", "Y", "D", "V"],
    "RRMSE_error": [
        X_RRMSE,
        Y_RRMSE,
        D_RRMSE,
        V_RRMSE
    ]
})

print("RRMSE errors:")
print(rrmse_df)